# rpfr-visualizer — a tour

Everything here runs against HDF5 files on your own machine. Nothing is
uploaded, and the browser app in §8 listens on `127.0.0.1` only.

Read the next cell before running anything: it is the only one you have to
edit.

**What you need**

```bash
uv sync --all-extras          # from a clone
pip install 'rpfr-visualizer[gui]'  # or from a release
```

and a folder of QM9s-RPFR HDF5 files. A search index is optional — without one
you can still open molecules by ID, but not search by formula, structure or
functional group.

## 1. Point it at your data

Three settings, and only the first is required.

| | |
|---|---|
| `DATA` | a single `.h5` file, a folder of them, or an external-drive root |
| `INDEX` | a `.sqlite` search index, or `None` to look in the usual places |
| `DATASET` | which dataset inside `DATA` to open, or `None` for the first |

Leave them as `None` to auto-discover: the repository's own `data/` folder,
then `$RPFR_VISUALIZER_DATA`, then `$RPFR_VISUALIZER_DRIVE`, then the drive's default
mount point.

In [ ]:
from __future__ import annotations

from pathlib import Path

# --- edit these three ------------------------------------------------------
DATA: Path | str | None = None
INDEX: Path | str | None = None
DATASET: str | None = None
# ---------------------------------------------------------------------------

# Worked examples, for reference:
#
#   DATA  = "data/release"                                   # a repo subsample
#   DATA  = "/path/to/Processed-Data" # the full corpus
#   INDEX = "data/cache/qm9s_rpfr_full_index.sqlite"
#   DATASET = "qm9s"

In [ ]:
import os

from rpfr_visualizer.io.molecule_index import default_drive_root
from rpfr_visualizer.io.search import find_index


def resolve_data() -> Path | None:
    """The first of the usual places that actually holds HDF5 files."""
    if DATA is not None:
        return Path(DATA).expanduser()

    here = Path.cwd().resolve()
    candidates = [root / "data" for root in (here, *here.parents[:3])]
    for var in ("RPFR_VISUALIZER_DATA", "RPFR_VISUALIZER_DRIVE"):
        if os.environ.get(var):
            candidates.insert(0, Path(os.environ[var]).expanduser())
    candidates.append(default_drive_root())

    for candidate in candidates:
        if not candidate.exists():
            continue
        if candidate.is_file() or any(candidate.rglob("*.h5")):
            return candidate
    return None


DATA_PATH = resolve_data()
INDEX_PATH = Path(INDEX).expanduser() if INDEX else (
    find_index(data_dir=DATA_PATH) if DATA_PATH else None
)

print("data :", DATA_PATH or "not found — set DATA above")
print("index:", INDEX_PATH or "none (open by ID works; search does not)")

## 2. Open it

`controller_for_path` is the one forgiving entry point: hand it a file, a
folder or a drive root and it works out the rest. It is the same object the
browser app drives, so anything you do here you can do there, and the other way
round.

In [ ]:
from rpfr_visualizer.viz.explorer.factory import controller_for_path

if DATA_PATH is None:
    raise SystemExit("Set DATA in the cell above and re-run.")

controller, notes = controller_for_path(
    DATA_PATH, dataset=DATASET, index_path=INDEX_PATH
)
for note in notes:
    print("·", note)

print()
print("datasets  :", ", ".join(controller.available_datasets()))
print("open      :", controller.dataset)
print("molecules :", f"{len(controller.provider.list_keys()):,}")

### The temperature grid

β is stored at thirteen temperatures, and **they are not in ascending order in
the file** — the two chemistry temperatures are appended after the sweep. Code
that sorts temperatures without carrying β along with them silently mismatches
every point, so the package keeps the stored order and offers a sorted view
separately.

In [ ]:
print("stored order :", [f"{t:g}" for t in controller.temperatures])
print("for a picker :", [f"{t:g}" for t in controller.temperatures_for_display])

## 3. One molecule

β — the reduced isotopic partition function ratio — is stored per atom. A site
with a large β concentrates the heavy isotope.

In [ ]:
key = next(
    (k for k in controller.provider.list_keys()[:50]
     if controller.provider.get_molecule_by_key(k).smiles),
    controller.provider.list_keys()[0],
)
controller.load(key)

molecule = controller.state.mol_a.molecule
print(f"{controller.dataset}:{key}   {molecule.smiles}   {molecule.n_atoms} atoms")
print()
print(f"{'site':>4}  {'element':<7}  {'beta @ 298.15 K':>16}")
for site in molecule.sites:
    if site.element == "H":
        continue
    print(f"{site.atom_index:>4}  {site.element:<7}  {site.beta_at(298.15):>16.6f}")

## 4. Temperatures, stored and otherwise

A stored temperature is **read**. Anything else is **fitted**, from ln β
against 1/T, and you choose the fit. The two answers are different in kind and
the package never blurs them: `BetaAtResult.interpolated` says which you got.

In [ ]:
from rpfr_visualizer.analysis.temperature import INTERPOLATION_METHODS, interpolate_beta

site = next(s for s in molecule.sites if s.element != "H")
temps, betas = site.temperatures, site.beta

stored = interpolate_beta(temps, betas, 298.15, warn=False)
fitted = interpolate_beta(temps, betas, 305.0, warn=False)
print(f"298.15 K -> {stored.value:.6f}   interpolated={stored.interpolated}")
print(f"305.00 K -> {fitted.value:.6f}   interpolated={fitted.interpolated}")

print()
print("The same off-grid temperature under each offered fit:")
for name, label in INTERPOLATION_METHODS:
    value = interpolate_beta(temps, betas, 305.0, warn=False, method=name).value
    print(f"  {name:<6} {label:<22} {value:.6f}")

### Where the fits disagree

They agree closely over most of the range and disagree wildly below 60 K,
because **nothing is stored between 20 K and 60 K**. No fit can be accurate
there; what a fit can be is physical. Degree 5 is the only offered method that
overshoots into that gap and returns a β that *rises* with temperature. The
default, a local quadratic through the four stored points around the
temperature, does not.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from rpfr_visualizer.analysis.temperature import beta_curve

fig, (wide, low) = plt.subplots(1, 2, figsize=(11, 4))
for name, label in INTERPOLATION_METHODS:
    grid, curve = beta_curve(temps, betas, num=400, method=name)
    for axis in (wide, low):
        axis.plot(grid, curve, lw=1.4, label=label)

order = np.argsort(temps)
for axis, title in ((wide, "the whole range"), (low, "the unconstrained gap")):
    axis.plot(np.asarray(temps)[order], np.asarray(betas)[order], "ko", ms=4,
              label="stored", zorder=5)
    axis.set_xlabel("temperature (K)")
    axis.set_ylabel("β")
    axis.set_title(title)
wide.set_yscale("log")
low.set_xlim(18, 62)
low.set_ylim(min(betas) * 0.9, max(betas) * 1.25)
wide.legend(fontsize=8)
fig.tight_layout()

## 5. Searching

One grammar, whether you type it in the browser or pass it here. A name, a
formula, a SMILES, a molecule ID, or an explicit `fg:` / `smarts:` prefix. Size
tokens are **AND**-ed onto the rest; commas are OR.

| query | means |
|---|---|
| `CCO` | that exact molecule (canonical SMILES match) |
| `fg:alcohol` | carries an alcohol group |
| `fg:alcohol&amine` | carries **both** |
| `smarts:[CX3](=O)[OX2H1]` | matches that SMARTS |
| `fg:alcohol, C:5-9` | alcohols with five to nine carbons |
| `qc:pass` | only molecules whose stored QC verdict passed |

In [ ]:
if controller.index is None:
    print("No search index, so this section needs one. Build it with:")
    print("  rpfr-visualizer index build --raw <folder> --out <index.sqlite>")
else:
    LIMIT = 200
    for query in ("fg:alcohol, C:5-9", "fg:alcohol&amine", "smarts:[CX3]=[OX1]"):
        hits = controller.search(query, limit=LIMIT)
        # A full count is a second query; this one stops at the cap, so say so
        # rather than printing a ceiling as though it were a total.
        count = f"{len(hits):>4}+" if len(hits) == LIMIT else f"{len(hits):>4} "
        example = f"  e.g. {hits[0].entry.key}" if hits else ""
        print(f"{query:<28} {count} hits{example}")

## 6. Clumping and the mass law

Two quantities that a per-atom colour cannot show, and that **exist only at the
stored temperatures** — unlike β they have no interpolation model, so an
off-grid temperature makes them unavailable rather than approximate.

Clumping is reported as 1000 ln C, in per mil.

In [ ]:
from rpfr_visualizer.analysis.clumping import compute_clumping
from rpfr_visualizer.analysis.models import AnalysisKind, AnalysisRequest

arrays = controller.state.mol_a.arrays
if arrays is None or getattr(arrays, "clumping", None) is None:
    print("This dataset stores no clumping arrays.")
else:
    result = compute_clumping(
        molecule, arrays, temperature=298.15, aggregation="mean"
    )
    matrix = result.pair_values
    labels = {s.atom_index: f"{s.element}{s.atom_index}" for s in molecule.sites}

    pairs = []
    for i in range(matrix.shape[0]):
        for j in range(i + 1, matrix.shape[1]):
            value = matrix[i, j]
            if np.isfinite(value):
                pairs.append((1000.0 * np.log(value), i, j))
    pairs.sort(key=lambda row: abs(row[0]), reverse=True)

    print("strongest pairs, 1000 ln C (per mil) at 298.15 K")
    for delta, i, j in pairs[:8]:
        print(f"  {labels.get(i, i):>5} · {labels.get(j, j):<5} {delta:>10.4f}")

## 7. Exporting

The CSV is one row per site. Give it several temperatures and you get one row
per site *per temperature*, with each site's series contiguous — the shape a
calibration fit wants. Every row records its own temperature, whether that
temperature was stored, and which fit produced it if not.

In [ ]:
from rpfr_visualizer.gui.download import download_rows

# The controller starts locked to carbon, as the browser does. Clear it so
# this example really is every site.
element_lock, controller.state.element = controller.state.element, None
controller.state.download_temperatures = (250.0, 273.15, 298.15)

rows = download_rows(controller, "molecule")

n_sites = len(molecule.sites)
print(f"{len(rows)} rows = {n_sites} sites x 3 temperatures")
print()
columns = ("atom_index", "element", "temperature_K", "on_grid", "beta_method", "beta")
print("  ".join(f"{c:>13}" for c in columns))
for row in rows[:6]:
    print("  ".join(f"{row[c]:>13}" for c in columns))

controller.state.download_temperatures = ()
controller.state.element = element_lock

## 8. The browser explorer

Same controller, same data, a picture instead of a table. This opens a local
server and your browser; the cell after it stops the server again.

From a terminal the equivalent is:

```bash
rpfr-visualizer gui --data <folder> --index <index.sqlite>
```

In [ ]:
from rpfr_visualizer.gui import start_explorer

SERVER, URL = start_explorer(
    data_dir=str(DATA_PATH),
    index_path=str(INDEX_PATH) if INDEX_PATH else None,
    dataset=DATASET,
    port=0,             # any free port
    open_browser=True,  # set False if you would rather open the link yourself
)
print("Explorer running at", URL)
print("Open that link, then run the next cell when you are done.")

In [ ]:
if "SERVER" in globals() and SERVER is not None:
    SERVER.shutdown()
    SERVER = None
    print("stopped")

## 9. Quality control

Every molecule carries a stored verdict aggregating several checks — whether
the frequency calculation converged, whether β is physical, whether
translations and rotations came out cleanly. Nothing is silently dropped: a
molecule that fails is still in the file, flagged.

In [ ]:
print(controller.qc_text())

## Where to go next

- `docs/gui.md` — the browser explorer in detail
- `docs/search.md` — the full query grammar
- `docs/schema.md` — the HDF5 v2.0 field reference
- `docs/dataset_qc_examples.ipynb` — the QC engine, in depth

Two things worth remembering:

1. **A stored temperature is read; anything else is fitted.** Between 20 K and
   60 K nothing is stored, so no fit is accurate there — prefer the grid.
2. **A subsample holds what it holds.** The published tier is 1% of QM9s, so a
   search for a familiar small molecule will usually find nothing.